# Comment Toxicity Detection — Training Notebook

This notebook walks through the full pipeline for training a **BiLSTM deep-learning model** to detect toxic comments across six categories: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, and `identity_hate`.

**Steps covered:**
1. Clone the project repository
2. Install required dependencies
3. Train the model (with GPU acceleration)
4. Package model artifacts for download
5. Deploy the Streamlit app via ngrok (optional)

> **Tip:** This notebook is designed to run on **Google Colab** with a T4 GPU runtime for optimal training speed.

## Step 1 — Clone the Repository

Clone the project from GitHub and navigate into the project directory. The repository includes the dataset, source code, and configuration files needed for training.

In [ ]:
# Cell 1: Clone the repository
!git clone https://github.com/10Unknownboy/Comment-Toxicity-Detection.git
%cd Comment-Toxicity-Detection

Cloning into 'Comment-Toxicity-Detection'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 116 (delta 58), reused 66 (delta 31), pack-reused 22 (from 1)
Receiving objects: 100% (116/116), 50.05 MiB | 12.27 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/Comment-Toxicity-Detection


## Step 2 — Install Dependencies

Install the required Python packages. Most of these come pre-installed on Google Colab, but running this cell ensures all versions are compatible.

In [ ]:
# Cell 2: Install dependencies
!pip install torch pandas numpy scikit-learn matplotlib tqdm

## Step 3 — Train the Model

Train the BiLSTM model on the Toxic Comment dataset. This cell:

- Loads and preprocesses the training data
- Builds a vocabulary from the corpus
- Trains the BiLSTM model for up to 20 epochs with early stopping
- Evaluates the best model on the held-out test set
- Tunes per-label decision thresholds to maximise F1 scores
- Saves all model artifacts to the `models/` directory

> **Note:** Using `--sample-size 0` trains on the full dataset (~560K comments). For a faster trial, use `--sample-size 160000`.

In [4]:
# Cell 3: Train the model (uses GPU automatically if available)
!python -m src.train --epochs 20 --batch-size 256 --sample-size 0

08:59:31 [INFO] ============================================================
08:59:31 [INFO]   Device : cuda
08:59:31 [INFO]   GPU    : Tesla T4
08:59:31 [INFO]   AMP    : enabled
08:59:31 [INFO] ============================================================
08:59:31 [INFO] [data] Loading training data from /content/Comment-Toxicity-Detection/data/train.csv …
08:59:32 [INFO] [data] Loaded 159,571 rows.
08:59:32 [INFO] [data] Cleaning texts …
08:59:45 [INFO] [data] Splits → train: 127,656  val: 15,957  test: 15,958
08:59:45 [INFO] [data] Augmenting rare classes …
08:59:48 [INFO]   threat: 404 → 2000 (augmenting 1596 samples)
08:59:48 [INFO]   identity_hate: 1111 → 2500 (augmenting 1389 samples)
08:59:48 [INFO]   Total augmented rows: 2985
08:59:48 [INFO] [data] After augmentation: 130,641 training samples.
08:59:48 [INFO] [data] Building vocabulary …
08:59:50 [INFO] [data] Vocabulary size: 50,000 tokens (+ PAD, UNK).
08:59:50 [INFO] [data] Vocabulary saved to /content/Comment-Toxicity-Det

## Step 4 — Package Model Artifacts

Zip all trained model files into a single archive for easy download. After running this cell, use the Colab file browser (left sidebar) to download `models.zip`.

In [5]:
# Cell 4: Zip model artifacts for easy download
!zip -r models.zip models/

  adding: models/ (stored 0%)
  adding: models/vocab.json (deflated 61%)
  adding: models/thresholds_precision.json (deflated 30%)
  adding: models/thresholds.json (deflated 33%)
  adding: models/roc_curves.png (deflated 12%)
  adding: models/evaluation_results.json (deflated 66%)
  adding: models/training_history.png (deflated 13%)
  adding: models/training_history.json (deflated 60%)
  adding: models/confusion_matrices.png (deflated 21%)
  adding: models/toxicity_model.pth (deflated 8%)


## Step 5 — Deploy with ngrok (Optional)

Launch the Streamlit web application directly from Colab using ngrok to create a public tunnel. This lets you preview the app without a local setup.

> **Important:** Replace the ngrok auth token below with your own token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) if needed.

In [ ]:
!pip install streamlit pyngrok --quiet

from pyngrok import ngrok # type: ignore
import threading
import os

# Paste your ngrok auth token here
ngrok.set_auth_token("<YOUR_NGROK_AUTH_TOKEN>")

def run():
    os.system("streamlit run app.py --server.port 8501")

threading.Thread(target=run).start()

public_url = ngrok.connect(8501)

print("Your Streamlit App is Live:")
print(public_url)

Your Streamlit App is Live:
NgrokTunnel: "https://1bec-34-13-202-174.ngrok-free.app" -> "http://localhost:8501"
